# Footfall Analysis - Unified Inference

Runs **gender** (ViT, HuggingFace) and **age** (SVM, HuggingFace) predictions on the same input video in a single pipeline.

**Per-frame pipeline:**
1. Detect persons with Faster R-CNN
2. For each crop: Gender ViT -> `male / female`
3. For each crop: Region color+LBP features -> Age SVM -> `Age16-30 / Age31-45 / Age46-60 / AgeAbove61`
4. Draw annotated bounding boxes and write output video


## 1. Install Dependencies

In [ ]:
# Run once (Colab / fresh env)
!pip install -q huggingface_hub transformers torch torchvision scikit-image scikit-learn opencv-python-headless Pillow tqdm scipy


## 2. Configuration

In [ ]:
# HuggingFace repos
GENDER_REPO  = 'abhshkp/footfall-analysis-vit-stage2'
AGE_REPO     = 'abhshkp/footfall-analysis-age-svm'
AGE_PKL_FILE = 'age_classifier_bundle.pkl'

# Output video path
OUTPUT_VIDEO_PATH = 'output_annotated.mp4'

# Detection / classification thresholds
DETECTION_SCORE_THRESH = 0.70   # Faster R-CNN person confidence
GENDER_CONF_THRESH     = 0.55   # below this softmax conf -> 'unknown'
MIN_CROP_PX            = 20     # skip crops smaller than this (px)
COCO_PERSON_CLASS_ID   = 1

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)


## 2a. Input Video

- **Colab**: the cell below opens a file-picker so you can upload a video directly.
- **Local**: just set `INPUT_VIDEO_PATH` to your video file path.


In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    # -- Colab: open file upload dialog ----------------------------------
    from google.colab import files
    print('Upload your input video (mp4 / avi / mov / mkv):')
    uploaded = files.upload()           # opens the Colab file-picker
    if not uploaded:
        raise RuntimeError('No file uploaded. Re-run this cell and select a video.')
    INPUT_VIDEO_PATH = list(uploaded.keys())[0]
    print('Input video set to:', INPUT_VIDEO_PATH)
else:
    # -- Local: set path manually ----------------------------------------
    INPUT_VIDEO_PATH = 'input_video.mp4'   # <-- change this
    if not os.path.exists(INPUT_VIDEO_PATH):
        print('WARNING: file not found -', INPUT_VIDEO_PATH)
    else:
        print('Input video set to:', INPUT_VIDEO_PATH)


## 3. Imports

In [ ]:
import pickle, json, warnings
import numpy as np
import cv2
from pathlib import Path
from scipy.spatial.distance import cdist
from PIL import Image
from skimage.feature import local_binary_pattern
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms.functional as TF
from transformers import ViTModel, ViTImageProcessor
from huggingface_hub import hf_hub_download, snapshot_download

warnings.filterwarnings('ignore')
print('Imports OK')


## 4. Load Gender Model (ViT from HuggingFace)

In [ ]:
class GenderClassifier(nn.Module):
    def __init__(self, vit_model_name='google/vit-base-patch16-224', num_labels=2):
        super().__init__()
        self.vit = ViTModel.from_pretrained(vit_model_name)
        self.head = nn.Linear(self.vit.config.hidden_size, num_labels)
        self.id2label = {0: 'female', 1: 'male'}
        self.label2id = {'female': 0, 'male': 1}

    def forward(self, pixel_values):
        out = self.vit(pixel_values=pixel_values)
        return self.head(out.pooler_output)

    @classmethod
    def load(cls, ckpt_dir, device='cpu'):
        with open(Path(ckpt_dir) / 'config.json') as f:
            cfg = json.load(f)
        m = cls(cfg['vit_model_name'], cfg.get('num_labels', 2))
        state = torch.load(
            Path(ckpt_dir) / 'pytorch_model.bin',
            map_location=device, weights_only=True,
        )
        m.load_state_dict(state, strict=False)
        return m.to(device).eval()


print('Downloading gender model from', GENDER_REPO, '...')
gender_ckpt_dir  = snapshot_download(GENDER_REPO)
gender_model     = GenderClassifier.load(gender_ckpt_dir, device=DEVICE)
gender_processor = ViTImageProcessor.from_pretrained(gender_ckpt_dir)
print('Gender model loaded')


## 5. Load Age Model (SVM bundle from HuggingFace)

In [ ]:
print('Downloading age SVM bundle from', AGE_REPO, '...')
pkl_path = hf_hub_download(repo_id=AGE_REPO, filename=AGE_PKL_FILE)

with open(pkl_path, 'rb') as f:
    bundle = pickle.load(f)

# Bundle keys: classifiers, configs, hyperparams, X_train, y_train,
#              AGE_CLASSES, AGE_TO_IDX, NUM_CLASSES, val_scores, meta
age_classifiers = bundle['classifiers']
age_configs     = bundle['configs']
X_train_age     = bundle['X_train']     # needed for intersection kernel inference
AGE_CLASSES     = bundle['AGE_CLASSES']
hp              = bundle['hyperparams']

N_REGIONS  = hp['N_REGIONS']
COLOR_BINS = hp['COLOR_BINS']
LBP_POINTS = hp['LBP_POINTS']
LBP_RADII  = hp['LBP_RADII']

print('Age classifiers:', list(age_classifiers.keys()))
print('Kernel configs: ', age_configs)
print('Feature params: N_REGIONS={}, COLOR_BINS={}, LBP_POINTS={}, LBP_RADII={}'.format(
    N_REGIONS, COLOR_BINS, LBP_POINTS, LBP_RADII))
print('X_train shape: ', X_train_age.shape)


## 6. Load Person Detector (Faster R-CNN)

In [ ]:
print('Loading Faster R-CNN ...')
detector = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights='DEFAULT')
detector.eval().to(DEVICE)
print('Detector loaded')


## 7. Feature Extraction Helpers (mirror of training pipeline)

In [ ]:
def region_split(img, n_regions=N_REGIONS):
    w, h = img.size
    step = h // n_regions
    regions = []
    for i in range(n_regions):
        top    = i * step
        bottom = (i + 1) * step if i < n_regions - 1 else h
        regions.append(img.crop((0, top, w, bottom)))
    return regions


def color_histogram(region, bins=COLOR_BINS):
    rgb = np.array(region.convert('RGB')).astype(np.float32)
    hsv = np.array(region.convert('HSV')).astype(np.float32)
    hist = []
    for c in range(3):   # R, G, B
        h, _ = np.histogram(rgb[:, :, c], bins=bins, range=(0, 255))
        hist.append(h / (h.sum() + 1e-8))
    for c in range(2):   # H, S
        h, _ = np.histogram(hsv[:, :, c], bins=bins, range=(0, 255))
        hist.append(h / (h.sum() + 1e-8))
    return np.concatenate(hist)


def texture_histogram(region, points=LBP_POINTS, radii=None):
    if radii is None:
        radii = LBP_RADII
    gray = np.array(region.convert('L'))
    hists = []
    for r in radii:
        lbp = local_binary_pattern(gray, points, r, method='uniform')
        n_bins = points + 2
        h, _ = np.histogram(lbp, bins=n_bins, range=(0, n_bins))
        hists.append(h / (h.sum() + 1e-8))
    return np.concatenate(hists)


def extract_features_from_pil(pil_img):
    regions = region_split(pil_img)
    feats = []
    for r in regions:
        feats.append(color_histogram(r))
        feats.append(texture_histogram(r))
    return np.concatenate(feats)


def intersection_kernel(X, Y):
    sums_x = X.sum(axis=1)
    sums_y = Y.sum(axis=1)
    l1 = cdist(X, Y, metric='cityblock')
    return 0.5 * (sums_x[:, None] + sums_y[None, :]) - 0.5 * l1


print('Feature helpers ready')


## 8. Inference Functions

In [ ]:
@torch.no_grad()
def detect_persons(frame_rgb):
    img_t = TF.to_tensor(Image.fromarray(frame_rgb)).to(DEVICE)
    outputs = detector([img_t])[0]
    boxes = []
    for box, label, score in zip(outputs['boxes'], outputs['labels'], outputs['scores']):
        if label.item() == COCO_PERSON_CLASS_ID and score.item() >= DETECTION_SCORE_THRESH:
            x1, y1, x2, y2 = box.cpu().numpy().astype(int)
            boxes.append((x1, y1, x2, y2))
    return boxes


@torch.no_grad()
def predict_gender(crop_pil):
    inputs = gender_processor(images=crop_pil, return_tensors='pt').to(DEVICE)
    logits = gender_model(inputs['pixel_values'])
    probs  = F.softmax(logits, dim=-1).squeeze()
    conf, idx = probs.max(dim=-1)
    label = gender_model.id2label[idx.item()]
    if conf.item() < GENDER_CONF_THRESH:
        label = 'unknown'
    return label, conf.item()


def predict_age(crop_pil):
    feats = extract_features_from_pil(crop_pil).reshape(1, -1)
    best_score, best_label = -1e9, None
    for cls in AGE_CLASSES:
        kernel_type = age_configs[cls][0]
        if kernel_type == 'intersection':
            k = intersection_kernel(feats, X_train_age)
            score = age_classifiers[cls].decision_function(k)[0]
        else:
            score = age_classifiers[cls].decision_function(feats)[0]
        if score > best_score:
            best_score, best_label = score, cls
    return best_label


print('Inference functions ready')


## 9. Annotation Helpers

In [ ]:
GENDER_COLOR = {
    'male':    (219, 152,  52),
    'female':  (147,  20, 255),
    'unknown': (128, 128, 128),
}
AGE_COLOR = {
    'Age16-30':   (  0, 200,  80),
    'Age31-45':   (  0, 160, 255),
    'Age46-60':   (255, 140,   0),
    'AgeAbove61': (180,   0, 200),
}


def draw_annotation(frame_bgr, x1, y1, x2, y2, gender_label, gender_conf, age_label):
    g_color = GENDER_COLOR.get(gender_label, (128, 128, 128))
    a_color = AGE_COLOR.get(age_label, (255, 255, 255))
    cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), g_color, 2)

    font, scale, thickness, pad = cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1, 3
    gender_pct  = int(gender_conf * 100)
    gender_text = '{} {}%'.format(gender_label.capitalize(), gender_pct)
    age_text    = age_label

    (gw, gh), _ = cv2.getTextSize(gender_text, font, scale, thickness)
    (aw, ah), _ = cv2.getTextSize(age_text,    font, scale, thickness)
    box_w = max(gw, aw) + 2 * pad
    box_h = gh + ah + 3 * pad + 4
    top   = max(y1 - box_h, 0)

    cv2.rectangle(frame_bgr, (x1, top), (x1 + box_w, y1), (30, 30, 30), -1)
    cv2.putText(frame_bgr, gender_text, (x1 + pad, top + gh + pad),
                font, scale, g_color, thickness, cv2.LINE_AA)
    cv2.putText(frame_bgr, age_text, (x1 + pad, top + gh + ah + 2 * pad + 4),
                font, scale, a_color, thickness, cv2.LINE_AA)


print('Annotation helpers ready')


## 10. Run Unified Inference on Video

In [ ]:
cap    = cv2.VideoCapture(INPUT_VIDEO_PATH)
fps    = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print('Video: {}x{} @ {:.1f} fps, {} frames'.format(width, height, fps, total))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (width, height))

frame_idx = total_detections = 0

with tqdm(total=total, desc='Processing frames', unit='fr') as pbar:
    while cap.isOpened():
        ret, frame_bgr = cap.read()
        if not ret:
            break

        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        boxes     = detect_persons(frame_rgb)

        for (x1, y1, x2, y2) in boxes:
            crop_arr = frame_rgb[y1:y2, x1:x2]
            if crop_arr.shape[0] < MIN_CROP_PX or crop_arr.shape[1] < MIN_CROP_PX:
                continue
            crop_pil = Image.fromarray(crop_arr)

            gender_label, gender_conf = predict_gender(crop_pil)
            age_label                 = predict_age(crop_pil)

            draw_annotation(frame_bgr, x1, y1, x2, y2,
                            gender_label, gender_conf, age_label)
            total_detections += 1

        writer.write(frame_bgr)
        frame_idx += 1
        pbar.update(1)

cap.release()
writer.release()
print('Done: {} frames, {} detections'.format(frame_idx, total_detections))
print('Output saved to:', OUTPUT_VIDEO_PATH)


## 11. Download Annotated Video

- **Colab**: runs automatically and triggers a browser download.
- **Local**: prints the absolute path to the output file.


In [ ]:
import os, sys
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    from google.colab import files
    print('Downloading', OUTPUT_VIDEO_PATH, '...')
    files.download(OUTPUT_VIDEO_PATH)
else:
    print('Output saved locally to:', os.path.abspath(OUTPUT_VIDEO_PATH))
